# Stage 06 Homework — Data Preprocessing

This notebook follows the provided starter: it creates the sample data,
applies reusable cleaning functions, records the assumptions, and saves a
verified processed dataset.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd

ROOT = Path.cwd()
if ROOT.name != "homework06":
    raise RuntimeError("Run this notebook from homework/homework06")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
raw_dir = ROOT / "data" / "raw"
processed_dir = ROOT / "data" / "processed"
raw_dir.mkdir(parents=True, exist_ok=True)
processed_dir.mkdir(parents=True, exist_ok=True)
print("working from:", ROOT)

working from: /Users/hansonsun/Documents/ChatGPT/FRE Bootcamp/homework/homework06


## Generate the starter dataset

In [2]:
data = {
    "age": [34, 45, 29, 50, 38, np.nan, 41],
    "income": [55000, np.nan, 42000, 58000, np.nan, np.nan, 49000],
    "score": [0.82, 0.91, np.nan, 0.76, 0.88, 0.65, 0.79],
    "zipcode": ["90210", "10001", "60614", "94103", "73301", "12345", "94105"],
    "city": ["Beverly", "New York", "Chicago", "SF", "Austin", "Unknown", "San Francisco"],
    "extra_data": [np.nan, 42, np.nan, np.nan, np.nan, 5, np.nan],
}
raw = pd.DataFrame(data)
raw_path = raw_dir / "sample_data.csv"
raw.to_csv(raw_path, index=False)
print("raw dataset:", raw_path.relative_to(ROOT), raw.shape)
raw

raw dataset: data/raw/sample_data.csv (7, 6)


,age,income,score,zipcode,city,extra_data
0,34.0,55000.0,0.82,90210,Beverly,NaN
1,45.0,NaN,0.91,10001,New York,42.0
2,29.0,42000.0,NaN,60614,Chicago,NaN
3,50.0,58000.0,0.76,94103,SF,NaN
4,38.0,NaN,0.88,73301,Austin,NaN
5,NaN,NaN,0.65,12345,Unknown,5.0
6,41.0,49000.0,0.79,94105,San Francisco,NaN


## Inspect missingness before transformation

In [3]:
before = pd.DataFrame({
    "dtype": raw.dtypes.astype(str),
    "missing_count": raw.isna().sum(),
    "missing_fraction": raw.isna().mean(),
})
before

,dtype,missing_count,missing_fraction
age,float64,1,0.142857
income,float64,3,0.428571
score,float64,1,0.142857
zipcode,str,0,0.000000
city,str,0,0.000000
extra_data,float64,5,0.714286


## Apply the cleaning contract

Numeric gaps are median-filled because this small exercise provides no
defensible row-level predictor for imputation. `extra_data` is removed at
an explicit 50% threshold. Analysis columns are then min-max scaled;
identifier-like and categorical values are left unchanged.

In [4]:
from src.cleaning import drop_missing, fill_missing_median, normalize_data

numeric_columns = ["age", "income", "score"]
cleaned = fill_missing_median(raw, numeric_columns)
cleaned = drop_missing(cleaned, threshold=0.5)
cleaned = normalize_data(cleaned, numeric_columns)
cleaned

,age,income,score,zipcode,city
0,0.238095,0.8125,0.653846,90210,Beverly
1,0.761905,0.6250,1.000000,10001,New York
2,0.000000,0.0000,0.596154,60614,Chicago
3,1.000000,1.0000,0.423077,94103,SF
4,0.428571,0.6250,0.884615,73301,Austin
5,0.500000,0.6250,0.000000,12345,Unknown
6,0.571429,0.4375,0.538462,94105,San Francisco


## Save and compare

In [5]:
output_path = processed_dir / "sample_data_cleaned.csv"
cleaned.to_csv(output_path, index=False)
comparison = pd.DataFrame({
    "raw_missing": raw.isna().sum(),
    "cleaned_missing": cleaned.isna().sum().reindex(raw.columns),
    "retained": raw.columns.isin(cleaned.columns),
}).fillna({"cleaned_missing": 0}).astype({"cleaned_missing": int})
print("processed dataset:", output_path.relative_to(ROOT), cleaned.shape)
comparison

processed dataset: data/processed/sample_data_cleaned.csv (7, 5)


,raw_missing,cleaned_missing,retained
age,1,0,True
income,3,0,True
score,1,0,True
zipcode,0,0,True
city,0,0,True
extra_data,5,0,False


## Assumptions and tradeoffs

Median imputation assumes the observed distribution is representative of
the missing cases (roughly MCAR/MAR). If missing income is systematic, the
replacement can compress real differences. Dropping a mostly missing
column sacrifices possible rare information for a reproducible feature
contract. Min-max scaling aids comparison but is sensitive to future
extremes, so production fitting parameters would need to be learned on
training data only. The raw file remains untouched for auditability.

In [6]:
reloaded = pd.read_csv(output_path)
assert reloaded.shape == (7, 5)
assert not reloaded.isna().any().any()
assert "extra_data" not in reloaded
for column in numeric_columns:
    assert reloaded[column].between(0, 1).all()
print("All Stage 06 homework checks passed.")

All Stage 06 homework checks passed.
